# US Traffic Fatality Prediction — Data Profiling
**DATA 495: Data Science Capstone**  
**Carl Stolpe | UMGC | May 2026**

This notebook profiles the NHTSA FARS 2015–2016 dataset, documenting variable selection rationale, null rates, outliers, and data quality issues across the 20 key variables selected for modeling.

**Data source:** [US Traffic Fatality Records — Kaggle (NHTSA FARS)](https://www.kaggle.com/datasets/usdot/nhtsa-traffic-fatalities)

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 2. Load Raw FARS Tables

The FARS dataset is organized into three relational tables joined on `ST_CASE` + `YEAR`:
- **accident** — crash-level attributes (52 columns)
- **vehicle** — vehicle-level attributes (65 columns)
- **person** — person-level attributes (67 columns)

In [2]:
# Load accident and person tables for both years
# (see data/README.md for where these files come from and how to name them)
acc_2015 = pd.read_csv('../data/raw/accident_2015.csv', low_memory=False, encoding='latin1')
per_2015 = pd.read_csv('../data/raw/person_2015.csv',   low_memory=False, encoding='latin1')
acc_2016 = pd.read_csv('../data/raw/accident_2016.csv', low_memory=False, encoding='latin1')
per_2016 = pd.read_csv('../data/raw/person_2016.csv',   low_memory=False, encoding='latin1')

print(f'Accident 2015: {acc_2015.shape}')
print(f'Person 2015:   {per_2015.shape}')
print(f'Accident 2016: {acc_2016.shape}')
print(f'Person 2016:   {per_2016.shape}')

Accident 2015: (32538, 89)
Person 2015:   (81620, 129)
Accident 2016: (34748, 91)
Person 2016:   (86474, 129)


## 3. Join Tables and Filter to Drivers

In [3]:
def build_driver_dataset(acc_df, per_df, year):
    """Join accident and person tables, filter to drivers only."""
    acc_df = acc_df.copy()
    per_df = per_df.copy()
    acc_df['YEAR'] = year
    per_df['YEAR'] = year
    # The person file already repeats many crash-level columns (STATE, HOUR,
    # FUNC_SYS, RUR_URB, ...). Take only the accident-only columns so the
    # merge does not rename duplicates to HOUR_x / HOUR_y.
    acc_cols = ['ST_CASE', 'YEAR'] + [c for c in acc_df.columns if c not in per_df.columns]
    merged = per_df.merge(acc_df[acc_cols], on=['ST_CASE', 'YEAR'], how='inner')
    drivers = merged[merged['PER_TYP'] == 1].copy()
    return drivers

df_2015 = build_driver_dataset(acc_2015, per_2015, 2015)
df_2016 = build_driver_dataset(acc_2016, per_2016, 2016)
df_all  = pd.concat([df_2015, df_2016], ignore_index=True)

print(f'2015 driver records: {len(df_2015):,}')
print(f'2016 driver records: {len(df_2016):,}')
print(f'Combined:            {len(df_all):,}')

2015 driver records: 49,163
2016 driver records: 52,399
Combined:            101,562


## 4. Variable Selection

20 variables were selected from the 184 raw fields based on domain relevance (per NHTSA FARS Analytical User's Manual), null rates, and variance.

In [4]:
KEY_VARS = [
    'ST_CASE', 'STATE', 'MONTH', 'HOUR', 'LGT_COND', 'WEATHER',
    'FUNC_SYS', 'RUR_URB', 'MAN_COLL', 'FATALS', 'DRUNK_DR',
    'BODY_TYP', 'MOD_YEAR', 'SPEEDREL',
    'AGE', 'SEX', 'PER_TYP', 'INJ_SEV', 'REST_USE', 'ALC_RES', 'DR_DRINK'
]

profile_df = df_all[[v for v in KEY_VARS if v in df_all.columns]].copy()
print(f'Profile dataset shape: {profile_df.shape}')

Profile dataset shape: (101562, 19)


## 5. Null Rate Analysis

In [5]:
null_rates = (profile_df.isnull().sum() / len(profile_df) * 100).round(2)
null_summary = pd.DataFrame({
    'Missing Count': profile_df.isnull().sum(),
    'Missing %': null_rates,
    'Dtype': profile_df.dtypes
}).sort_values('Missing %', ascending=False)

print('Null Rate Summary')
print('-' * 40)
print(null_summary.to_string())

Null Rate Summary
----------------------------------------
          Missing Count  Missing %    Dtype
ST_CASE               0        0.0    int64
STATE                 0        0.0    int64
MONTH                 0        0.0    int64
HOUR                  0        0.0    int64
LGT_COND              0        0.0    int64
WEATHER               0        0.0    int64
FUNC_SYS              0        0.0    int64
RUR_URB               0        0.0    int64
MAN_COLL              0        0.0    int64
FATALS                0        0.0    int64
DRUNK_DR              0        0.0    int64
BODY_TYP              0        0.0  float64
MOD_YEAR              0        0.0  float64
AGE                   0        0.0    int64
SEX                   0        0.0    int64
PER_TYP               0        0.0    int64
INJ_SEV               0        0.0    int64
REST_USE              0        0.0    int64
ALC_RES               0        0.0    int64


## 6. Descriptive Statistics for Key Continuous Variables

In [6]:
cont_vars = ['AGE', 'FATALS', 'HOUR', 'MONTH', 'DRUNK_DR']

# Remap sentinel codes before computing stats
stats_df = profile_df[cont_vars].copy()
stats_df['HOUR'] = stats_df['HOUR'].replace(99, np.nan)
stats_df['AGE']  = stats_df['AGE'].replace([998, 999], np.nan)

desc = stats_df.describe().T
desc.index = ['Driver Age (years)', 'Fatalities per Crash',
              'Hour of Crash (0-23)', 'Month of Crash (1-12)',
              'Drunk Drivers in Crash']
desc.index.name = 'Variable'

print('Table 1: Descriptive Statistics — Key Continuous Variables (2015-2016 FARS, N=101,562)')
print(desc[['count','mean','std','min','25%','50%','75%','max']].round(2).to_string())

Table 1: Descriptive Statistics — Key Continuous Variables (2015-2016 FARS, N=101,562)
                           count   mean    std  min   25%   50%   75%    max
Variable                                                                    
Driver Age (years)       99421.0  42.73  18.24  5.0  27.0  40.0  56.0  103.0
Fatalities per Crash    101562.0   1.11   0.41  1.0   1.0   1.0   1.0   13.0
Hour of Crash (0-23)    101038.0  12.90   6.61  0.0   8.0  14.0  18.0   23.0
Month of Crash (1-12)   101562.0   6.78   3.35  1.0   4.0   7.0  10.0   12.0
Drunk Drivers in Crash  101562.0   0.27   0.47  0.0   0.0   0.0   1.0    3.0


## 7. Target Variable Distribution

In [7]:
inj_counts = profile_df['INJ_SEV'].value_counts().sort_index()
inj_labels = {
    0: 'No Injury',
    1: 'Possible Injury',
    2: 'Suspected Minor',
    3: 'Suspected Serious',
    4: 'Fatal Injury',
    5: 'Injured, Severity Unknown',
    6: 'Died Prior to Crash (dropped)',
    9: 'Unknown (dropped)'
}

print('INJ_SEV Distribution (pre-binary recoding)')
print('-' * 40)
for code, count in inj_counts.items():
    label = inj_labels.get(code, f'Code {code}')
    pct = count / len(profile_df) * 100
    print(f'  {label:<25} {count:>7,}  ({pct:.1f}%)')

fatal_count    = (profile_df['INJ_SEV'] == 4).sum()
nonfatal_count = (profile_df['INJ_SEV'] != 4).sum()
print(f'\nBinary target — Fatal (1): {fatal_count:,} ({fatal_count/len(profile_df):.1%})')
print(f'Binary target — Non-Fatal (0): {nonfatal_count:,} ({nonfatal_count/len(profile_df):.1%})')

INJ_SEV Distribution (pre-binary recoding)
----------------------------------------
  No Injury                  27,990  (27.6%)
  Possible Injury             7,814  (7.7%)
  Suspected Minor             9,653  (9.5%)
  Suspected Serious           7,973  (7.9%)
  Fatal Injury               46,065  (45.4%)
  Injured, Severity Unknown     233  (0.2%)
  Died Prior to Crash (dropped)       5  (0.0%)
  Unknown (dropped)           1,829  (1.8%)



Binary target — Fatal (1): 46,065 (45.4%)
Binary target — Non-Fatal (0): 55,497 (54.6%)


## 8. Sentinel Code Inventory

FARS uses numeric codes to represent unknown, not-reported, or inapplicable values (codes per the FARS Analytical User's Manual for 2015–2016). These must be remapped to NaN before modeling — otherwise a code like 99 ("unknown") is treated as a real, very large value.

In [8]:
sentinel_map = {
    'HOUR':     [99],
    'AGE':      [998, 999],
    'SEX':      [8, 9],
    'MOD_YEAR': [9998, 9999],
    'LGT_COND': [8, 9],
    'WEATHER':  [98, 99],
    'MAN_COLL': [98, 99],
    'FUNC_SYS': [96, 98, 99],
    'RUR_URB':  [6, 8, 9],
    'REST_USE': [96, 97, 98, 99],
    'INJ_SEV':  [6, 9],        # 6 = died prior to crash, 9 = unknown
    'ALC_RES':  [995, 996, 997, 998, 999],
}

print('Sentinel Code Inventory')
print('-' * 50)
for col, codes in sentinel_map.items():
    if col in profile_df.columns:
        count = profile_df[col].isin(codes).sum()
        pct   = count / len(profile_df) * 100
        print(f'  {col:<12} codes {codes} → {count:,} records ({pct:.1f}%)')

Sentinel Code Inventory
--------------------------------------------------
  HOUR         codes [99] → 524 records (0.5%)
  AGE          codes [998, 999] → 2,141 records (2.1%)
  SEX          codes [8, 9] → 2,013 records (2.0%)
  MOD_YEAR     codes [9998, 9999] → 2,337 records (2.3%)


  LGT_COND     codes [8, 9] → 340 records (0.3%)
  WEATHER      codes [98, 99] → 2,760 records (2.7%)
  MAN_COLL     codes [98, 99] → 318 records (0.3%)
  FUNC_SYS     codes [96, 98, 99] → 543 records (0.5%)
  RUR_URB      codes [6, 8, 9] → 384 records (0.4%)
  REST_USE     codes [96, 97, 98, 99] → 8,966 records (8.8%)
  INJ_SEV      codes [6, 9] → 1,834 records (1.8%)
  ALC_RES      codes [995, 996, 997, 998, 999] → 53,378 records (52.6%)


## 9. ALC_RES Coding — Feature Engineering Rationale

In the 2015–2016 FARS person file, `ALC_RES` is almost never blank. It is stored in **thousandths** (e.g. `80` = 0.080 g/dL), and values 995–999 are status codes rather than test results:

| Code | Meaning |
|------|---------|
| 0–939 | Actual BAC value in thousandths |
| 940 | 0.94 or greater |
| 995 | Not reported |
| 996 | Test not given |
| 997 | Test performed, results unknown |
| 998 | Positive reading with no actual value |
| 999 | Reported as unknown |

So "BAC ≥ 0.08" means `80 <= ALC_RES <= 940`, and "a test was given" means a value ≤ 940 or code 997/998.

**Leakage caution:** drivers who die are tested far more often than drivers who survive, so "was a test given" mostly reflects the outcome itself. `ALC_TESTED` is kept for profiling but is **not** used as a model feature.

In [9]:
if 'ALC_RES' in profile_df.columns:
    alc = profile_df['ALC_RES']
    print('ALC_RES status breakdown:')
    print(f'  Blank (NaN):                 {alc.isna().sum():,}')
    print(f'  Measured value (0-940):      {(alc <= 940).sum():,}')
    for code, label in [(995, 'Not reported'), (996, 'Test not given'),
                        (997, 'Tested, result unknown'), (998, 'Positive, no value'),
                        (999, 'Unknown')]:
        print(f'  {code} {label:<25} {(alc == code).sum():,}')

    tested   = (alc <= 940) | alc.isin([997, 998])
    positive = (alc >= 80) & (alc <= 940)
    fatal    = profile_df['INJ_SEV'] == 4
    print()
    print(f'Test given — fatally injured drivers: {tested[fatal].mean():.1%}')
    print(f'Test given — surviving drivers:       {tested[~fatal].mean():.1%}')
    print(f'BAC >= 0.08 among all drivers:        {positive.mean():.1%}')

ALC_RES status breakdown:
  Blank (NaN):                 0
  Measured value (0-940):      48,184
  995 Not reported              4,738
  996 Test not given            44,689
  997 Tested, result unknown    1,960
  998 Positive, no value        214
  999 Unknown                   1,777

Test given — fatally injured drivers: 74.7%
Test given — surviving drivers:       28.8%
BAC >= 0.08 among all drivers:        13.1%


## 10. Data Profile Summary

| Variable | Table | Type | Key Issue |
|----------|-------|------|----------|
| ST_CASE | Accident | ID | Combine with YEAR for uniqueness |
| STATE | Accident | Categorical | FIPS code — treat as a category, not a number |
| MONTH | Accident | Integer | Cyclical encoding recommended |
| HOUR | Accident | Integer | Code 99 = unknown → NaN |
| LGT_COND | Accident | Categorical | Codes 8, 9 = not reported/unknown |
| WEATHER | Accident | Categorical | Codes 98, 99 = not reported/unknown |
| FUNC_SYS | Accident | Categorical | Road class (1 = Interstate … 7 = Local); codes 96, 98, 99 = unknown. Not a rural/urban flag in 2015+ |
| RUR_URB | Accident | Categorical | 1 = Rural, 2 = Urban; codes 6, 8, 9 = unknown. Source of the rural/urban feature |
| MAN_COLL | Accident | Categorical | Code 0 = not a collision with a motor vehicle (valid); 98, 99 = unknown |
| FATALS | Accident | Count | Deaths in the crash — known only after the outcome, so excluded from modeling |
| DRUNK_DR | Accident | Count | Number of drinking drivers in the crash |
| AGE | Person | Integer | Codes 998, 999 → NaN |
| SEX | Person | Categorical | Codes 8, 9 → NaN |
| INJ_SEV | Person | Ordinal | Binary recoded: fatal (4) = 1, else 0; codes 6 and 9 dropped |
| REST_USE | Person | Categorical | Codes 96–99 = not applicable/other/unknown |
| ALC_RES | Person | Thousandths | 995–999 are status codes; ALC_POSITIVE = 80–940 |